# 04 Summary Jurnal — Seismic Hazard Analysis of Indonesia

## Cell 1: Setup, rcParams, Directories, Data Loading

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os; os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patches as patches
from matplotlib.colors import Normalize, BoundaryNorm
import matplotlib.cm as cm
from mpl_toolkits.axes_grid1 import make_axes_locatable
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pandas as pd, numpy as np
from pathlib import Path
from scipy import stats
from scipy.stats import pearsonr
from sklearn.metrics import silhouette_score

# rcParams (Arial with fallback)
plt.rcParams.update({
    'font.family': ['Arial', 'DejaVu Sans'],
    'font.size': 11,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.dpi': 100,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

WORKDIR = Path(".")
FIGDIR  = WORKDIR / 'figures'
TABDIR  = WORKDIR / 'tables'
FIGDIR.mkdir(exist_ok=True)
TABDIR.mkdir(exist_ok=True)

ZONES = ['Zona Sumatera', 'Zona Jawa-Bali-NTB', 'Zona Sulawesi-NTT', 'Zona Maluku', 'Zona Papua']
ZSHORT = {
    'Zona Sumatera': 'Sumatera',
    'Zona Jawa-Bali-NTB': 'Jawa-Bali-NTB',
    'Zona Sulawesi-NTT': 'Sulawesi-NTT',
    'Zona Maluku': 'Maluku',
    'Zona Papua': 'Papua',
}
ZCOLORS = {
    'Zona Sumatera': '#E74C3C',
    'Zona Jawa-Bali-NTB': '#3498DB',
    'Zona Sulawesi-NTT': '#2ECC71',
    'Zona Maluku': '#F39C12',
    'Zona Papua': '#9B59B6',
}

# Load all data
df = pd.read_csv(WORKDIR / 'indonesia_earthquakes_clustered.csv')
df['time'] = pd.to_datetime(df['time'], format='mixed', errors='coerce')
summary  = pd.read_csv(WORKDIR / 'output_spatio_temporal' / 'spatio_temporal_summary.csv')
gaps_df  = pd.read_csv(WORKDIR / 'output_spatio_temporal' / 'seismic_gap_zones.csv')
metrics  = pd.read_csv(WORKDIR / 'lstm_metrics_v3.csv')
forecast = pd.read_csv(WORKDIR / 'lstm_forecast_2026_2027_v2.csv')

preds = {}
for z in ZONES:
    zfn = z.replace(' ', '_').replace('-', '_')
    fp = WORKDIR / 'output_lstm' / f'predictions_{zfn}_v3.csv'
    p = pd.read_csv(fp)
    p['date'] = pd.to_datetime(p['month'])
    preds[z] = p

print(f'Dataset loaded: {len(df):,} events')
print(f'Figures dir: {FIGDIR}')
print(f'Tables  dir: {TABDIR}')
for z in ZONES:
    n = (df['zone_name'] == z).sum()
    print(f'  {z}: {n:,} events')

## Cell 2: Figure 1 — Seismicity Map (cartopy)

In [ ]:
# Uses: df, ZONES, ZCOLORS, ZSHORT
# Scatter colored by depth (colorbar), size by mag
# Zone labels at centroids, legend, coastlines
# Cities: Jakarta(-6.21,106.85), Surabaya(-7.25,112.75), Makassar(-5.14,119.42), Jayapura(-2.53,140.72)

fig = plt.figure(figsize=(7.48, 5.0))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
ax.set_extent([94, 142, -12, 10], crs=ccrs.PlateCarree())

ax.add_feature(cfeature.OCEAN, facecolor='#D6EAF8', zorder=0)
ax.add_feature(cfeature.LAND,  facecolor='#F0F0E8', zorder=1)
ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='#555555', zorder=2)
ax.add_feature(cfeature.BORDERS, linewidth=0.4, edgecolor='#888888', linestyle='--', zorder=2)

# scatter by depth
sc = ax.scatter(df['longitude'], df['latitude'],
                c=df['depth'], cmap='plasma_r', vmin=0, vmax=700,
                s=df['mag']**2 * 0.3, alpha=0.3,
                transform=ccrs.PlateCarree(), rasterized=True, zorder=3)

cbar = plt.colorbar(sc, ax=ax, orientation='vertical', pad=0.02, shrink=0.7, aspect=20)
cbar.set_label('Depth (km)', fontsize=9)
cbar.ax.tick_params(labelsize=8)

# Zone legend patches
handles = [mpatches.Patch(facecolor=ZCOLORS[z], label=ZSHORT[z], alpha=0.85) for z in ZONES]
ax.legend(handles=handles, loc='upper left', fontsize=8, title='Seismic Zone',
          title_fontsize=8, framealpha=0.9)

# Zone centroids labels
for z in ZONES:
    sub = df[df['zone_name'] == z]
    cx = float(sub['longitude'].median())
    cy = float(sub['latitude'].median())
    ax.text(cx, cy, ZSHORT[z], transform=ccrs.PlateCarree(),
            fontsize=7, fontweight='bold', color=ZCOLORS[z],
            ha='center', va='center', zorder=5,
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7, edgecolor=ZCOLORS[z], lw=0.8))

# Cities (Medan removed)
cities = {'Jakarta': (-6.21, 106.85), 'Surabaya': (-7.25, 112.75),
          'Makassar': (-5.14, 119.42), 'Jayapura': (-2.53, 140.72)}
for city, (lat, lon) in cities.items():
    ax.plot(lon, lat, 'k^', ms=4, transform=ccrs.PlateCarree(), zorder=6)
    ax.text(lon+0.3, lat+0.3, city, transform=ccrs.PlateCarree(), fontsize=7,
            color='#222222', zorder=6)

# North arrow
ax.annotate('N', xy=(0.97, 0.88), xytext=(0.97, 0.80),
            xycoords='axes fraction', fontsize=10, fontweight='bold', ha='center',
            arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

# Scale bar at bottom-left (approx 500 km)
ax.plot([96, 100.5], [-11.2, -11.2], 'k-', lw=2, transform=ccrs.PlateCarree())
ax.text(98.25, -10.7, '~500 km', transform=ccrs.PlateCarree(), fontsize=7, ha='center')

# Gridlines
gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 8}
gl.ylabel_style = {'size': 8}

ax.set_title('Seismicity Map of Indonesia (1950-2026)\nN=94,902 events, colored by focal depth',
             fontsize=11, fontweight='bold', pad=6)

plt.tight_layout()
plt.savefig(FIGDIR / 'figure1_seismicity_map.png', dpi=300)
plt.show()
print('[OK] figure1_seismicity_map.png')

## Cell 3: Figure 2 — Temporal Analysis (GridSpec 3x2)

In [ ]:
# For each zone: bar chart monthly count (gray), rolling mean 12mo (colored line)
# Highlight shading 1950-1960 and 2020-2026
# Vertical red lines for M>=7.0 events

TRAIN_END = pd.Timestamp('2019-12-01')
VAL_END   = pd.Timestamp('2022-12-01')

date_range = pd.period_range('1950-01', '2026-05', freq='M')

ts_all = {}
for z in ZONES:
    sub = df[df['zone_name'] == z].copy()
    sub['ym'] = sub['time'].dt.to_period('M')
    grp = sub.groupby('ym').agg(monthly_count=('mag','count')).reindex(date_range, fill_value=0)
    grp.index = grp.index.to_timestamp()
    ts_all[z] = grp

# Find large events per zone (M>=7.0)
large_events = df[df['mag'] >= 7.0].copy()
large_events['year'] = large_events['time'].dt.year

fig = plt.figure(figsize=(7.48, 9.0))
from matplotlib.gridspec import GridSpec
gs = GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.3)
axes = [
    fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]),
    fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1]),
    fig.add_subplot(gs[2, :]),
]

for i, z in enumerate(ZONES):
    ax  = axes[i]
    ts  = ts_all[z]
    col = ZCOLORS[z]
    roll = ts['monthly_count'].rolling(12, min_periods=6).mean()

    ax.bar(ts.index, ts['monthly_count'], width=25, color='#AAAAAA', alpha=0.5, label='Monthly count')
    ax.plot(ts.index, roll, color=col, lw=1.8, alpha=0.9, label='12-mo MA')

    # Highlight periods
    ax.axvspan(pd.Timestamp('1950-01-01'), pd.Timestamp('1960-12-31'), alpha=0.08, color='blue', label='1950-1960')
    ax.axvspan(pd.Timestamp('2020-01-01'), pd.Timestamp('2026-05-31'), alpha=0.08, color='red', label='2020-2026')

    # Train/val/test boundaries
    ax.axvline(TRAIN_END, color='green', ls=':', lw=0.8, alpha=0.7)
    ax.axvline(VAL_END,   color='orange', ls=':', lw=0.8, alpha=0.7)

    # Large event annotations
    z_large = large_events[large_events['zone_name'] == z]
    for _, ev in z_large.iterrows():
        if ev['time'] is not pd.NaT:
            ax.axvline(ev['time'], color='red', lw=0.6, alpha=0.6)
            if ev['mag'] >= 7.5:
                ax.text(ev['time'], ax.get_ylim()[1]*0.85 if ax.get_ylim()[1]>0 else 5,
                        f"M{ev['mag']:.1f}", fontsize=5.5, color='red',
                        rotation=90, ha='right', va='top')

    ax.set_title(ZSHORT[z], fontsize=10, color=col, fontweight='bold')
    ax.set_ylabel('Events/month', fontsize=8)
    ax.set_xlim(pd.Timestamp('1950-01-01'), pd.Timestamp('2026-06-01'))
    ax.grid(axis='y', alpha=0.25)
    if i == 4:
        ax.legend(fontsize=7, ncol=4, loc='upper left')

fig.suptitle('Monthly Seismicity Time Series per Zone (1950-2026)', fontsize=12, fontweight='bold', y=0.98)
plt.savefig(FIGDIR / 'figure2_temporal_analysis.png', dpi=300)
plt.show()
print('[OK] figure2_temporal_analysis.png')

In [ ]:
# Figure 2 — 5 separate single-panel files (190mm x 120mm each)
from mpl_toolkits.axes_grid1.inset_locator import inset_axes as _inset_axes  # noqa

ZONE_FNAMES = {
    'Zona Sumatera':      'figure2a_temporal_sumatera',
    'Zona Jawa-Bali-NTB': 'figure2b_temporal_jawa_bali_ntb',
    'Zona Sulawesi-NTT':  'figure2c_temporal_sulawesi_ntt',
    'Zona Maluku':        'figure2d_temporal_maluku',
    'Zona Papua':         'figure2e_temporal_papua',
}

for z in ZONES:
    ts   = ts_all[z]
    col  = ZCOLORS[z]
    roll = ts['monthly_count'].rolling(12, min_periods=6).mean()

    fig2, ax2 = plt.subplots(figsize=(7.48, 4.72))

    ax2.bar(ts.index, ts['monthly_count'], width=25, color='#AAAAAA', alpha=0.5, label='Monthly count')
    ax2.plot(ts.index, roll, color=col, lw=1.8, alpha=0.9, label='12-mo MA')

    ax2.axvspan(pd.Timestamp('1950-01-01'), pd.Timestamp('1960-12-31'),
                alpha=0.08, color='blue', label='1950-1960')
    ax2.axvspan(pd.Timestamp('2020-01-01'), pd.Timestamp('2026-05-31'),
                alpha=0.08, color='red',  label='2020-2026')

    z_large = large_events[large_events['zone_name'] == z]
    for _, ev in z_large.iterrows():
        if ev['time'] is not pd.NaT:
            ax2.axvline(ev['time'], color='red', lw=0.6, alpha=0.6)
            if ev['mag'] >= 7.5:
                ymax = ax2.get_ylim()[1] if ax2.get_ylim()[1] > 0 else 5
                ax2.text(ev['time'], ymax * 0.85, f"M{ev['mag']:.1f}",
                         fontsize=5.5, color='red', rotation=90, ha='right', va='top')

    ax2.set_title(f'Monthly Seismicity — {ZSHORT[z]} (1950–2026)',
                  fontsize=11, fontweight='bold', color=col)
    ax2.set_ylabel('Events/month', fontsize=9)
    ax2.set_xlabel('Year', fontsize=9)
    ax2.set_xlim(pd.Timestamp('1950-01-01'), pd.Timestamp('2026-06-01'))
    ax2.grid(axis='y', alpha=0.25)
    ax2.legend(fontsize=8, ncol=4, loc='upper left')

    plt.tight_layout()
    fname = ZONE_FNAMES[z]
    plt.savefig(FIGDIR / f'{fname}.png', dpi=300)
    plt.close(fig2)
    print(f'[OK] {fname}.png')

print('\nFigure 2 individual files:')
for z in ZONES:
    fp = FIGDIR / f'{ZONE_FNAMES[z]}.png'
    if fp.exists():
        print(f'  {fp.name}: {fp.stat().st_size / 1024:.1f} KB')

## Cell 4: Figure 3 — Gutenberg-Richter (5 panels, 1x5)

In [ ]:
# For each zone: compute cumulative frequency-magnitude
# Mc via maximum curvature method
# Linear regression above Mc -> b-value, CI shading

def compute_gr_params(mags, bin_size=0.1):
    mags = np.asarray(mags)
    mags = mags[~np.isnan(mags) & (mags > 0)]
    m_min = np.floor(mags.min() * 10) / 10
    m_max = np.ceil(mags.max()  * 10) / 10
    bins  = np.arange(m_min, m_max + bin_size, bin_size)
    counts, edges = np.histogram(mags, bins=bins)
    centers = edges[:-1] + bin_size / 2

    # Mc: maximum curvature
    if counts.max() == 0:
        mc_idx = 0
    else:
        mc_idx = np.argmax(counts)
    mc = centers[mc_idx]

    # Cumulative N >= M
    cum_N = np.array([np.sum(mags >= m) for m in edges[:-1]])
    valid  = cum_N > 0
    M_all  = edges[:-1][valid]
    cN_all = cum_N[valid]

    # Fit above Mc
    fit_mask = (edges[:-1] >= mc) & (cum_N > 0)
    if fit_mask.sum() < 3:
        fit_mask = valid
    M_fit  = edges[:-1][fit_mask]
    logN_f = np.log10(cum_N[fit_mask])

    slope, intercept, r_val, p_val, std_err = stats.linregress(M_fit, logN_f)
    b = -slope
    a = intercept

    # CI (95%): t-distribution
    n = len(M_fit)
    t_val = stats.t.ppf(0.975, df=n-2) if n > 2 else 1.96
    se = std_err
    M_pred = np.linspace(M_fit.min(), M_fit.max(), 100)
    logN_pred = a - b * M_pred
    ci = t_val * se * np.sqrt(1/n + (M_pred - M_fit.mean())**2 / np.sum((M_fit - M_fit.mean())**2)) if n > 2 else np.zeros(100)

    r2 = r_val**2

    return dict(mc=mc, b=b, a=a, r2=r2, std_err=std_err,
                M_all=M_all, logcN_all=np.log10(cN_all),
                M_fit=M_fit, logN_fit=logN_f,
                M_pred=M_pred, logN_pred=logN_pred, ci=ci,
                n_events=len(mags))

fig, axes = plt.subplots(1, 5, figsize=(7.48, 3.2), sharey=False)
fig.suptitle('Gutenberg-Richter Frequency-Magnitude Distribution per Zone',
             fontsize=10, fontweight='bold', y=1.01)

for i, z in enumerate(ZONES):
    ax  = axes[i]
    col = ZCOLORS[z]
    sub = df[df['zone_name'] == z]
    gr  = compute_gr_params(sub['mag'].values)

    ax.scatter(gr['M_all'], gr['logcN_all'], s=14, color=col, alpha=0.75, zorder=3, label='Observed')
    ax.plot(gr['M_pred'], gr['logN_pred'], color=col, lw=1.5, zorder=4)
    ax.fill_between(gr['M_pred'],
                    gr['logN_pred'] - gr['ci'],
                    gr['logN_pred'] + gr['ci'],
                    color=col, alpha=0.15, label='95% CI')
    ax.axvline(gr['mc'], color='gray', ls='--', lw=1, alpha=0.7)

    b_str  = f"b = {gr['b']:.3f}"
    r2_str = f"R2 = {gr['r2']:.3f}"
    mc_str = f"Mc = {gr['mc']:.1f}"
    ax.text(0.95, 0.95, b_str,  transform=ax.transAxes, fontsize=7.5,
            ha='right', va='top', fontweight='bold', color=col)
    ax.text(0.95, 0.85, r2_str, transform=ax.transAxes, fontsize=7, ha='right', va='top')
    ax.text(0.95, 0.75, mc_str, transform=ax.transAxes, fontsize=7, ha='right', va='top', color='gray')

    ax.set_xlabel('Magnitude', fontsize=8)
    if i == 0:
        ax.set_ylabel('log10(N >= M)', fontsize=8)
    ax.set_title(ZSHORT[z], fontsize=9, color=col, fontweight='bold')
    ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig(FIGDIR / 'figure3_gutenberg_richter.png', dpi=300)
plt.show()
print('[OK] figure3_gutenberg_richter.png')

## Cell 5: Figure 4 — Seismic Gap Map (cartopy)

In [ ]:
# Parse gap locations, draw rectangles, top 10 with thick border
# Zone subduksi labels

def parse_gap_loc(loc_str):
    # "LAT7_LON95" -> (7, 95), "LAT-4_LON113" -> (-4, 113)
    parts = loc_str.strip().split('_')
    lat = int(parts[0].replace('LAT', ''))
    lon = int(parts[1].replace('LON', ''))
    return lat, lon

spatial_gaps = gaps_df[gaps_df['gap_type'] == 'spatial'].copy()
spatial_gaps['lat'] = spatial_gaps['location'].apply(lambda x: parse_gap_loc(x)[0])
spatial_gaps['lon'] = spatial_gaps['location'].apply(lambda x: parse_gap_loc(x)[1])

# Gap score proxy: number of historical events in description
import re
def extract_hist_events(desc):
    m = re.search(r'hist: (\d+) events', str(desc))
    return int(m.group(1)) if m else 1

spatial_gaps['hist_events'] = spatial_gaps['description'].apply(extract_hist_events)
max_hist = spatial_gaps['hist_events'].max()

fig = plt.figure(figsize=(7.48, 5.0))
ax  = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
ax.set_extent([94, 142, -12, 10], crs=ccrs.PlateCarree())

ax.add_feature(cfeature.OCEAN, facecolor='#D6EAF8', zorder=0)
ax.add_feature(cfeature.LAND,  facecolor='#F5F5F0', zorder=1)
ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='#444444', zorder=2)
ax.add_feature(cfeature.BORDERS, linewidth=0.3, edgecolor='#888888', linestyle='--', zorder=2)

# Background scatter (low density)
ax.scatter(df['longitude'][::5], df['latitude'][::5],
           s=0.3, c='#CCCCCC', alpha=0.3, transform=ccrs.PlateCarree(), rasterized=True, zorder=2)

# Sort gaps: top 10 by hist_events
spatial_gaps_sorted = spatial_gaps.sort_values('hist_events', ascending=False).reset_index(drop=True)
top10 = set(spatial_gaps_sorted.index[:10].tolist())

gap_norm = Normalize(vmin=1, vmax=max_hist)
cmap_gap = cm.get_cmap('Reds')

for idx, row in spatial_gaps_sorted.iterrows():
    lat, lon = row['lat'], row['lon']
    score    = row['hist_events']
    color    = cmap_gap(gap_norm(score))
    lw       = 2.5 if idx in top10 else 0.8
    # Draw 1x1 degree box
    rect = mpatches.Rectangle((lon, lat), 1, 1, linewidth=lw,
                               edgecolor='#C0392B', facecolor=color, alpha=0.65,
                               transform=ccrs.PlateCarree(), zorder=4)
    ax.add_patch(rect)

# Colorbar for gap score
sm = cm.ScalarMappable(cmap=cmap_gap, norm=gap_norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, orientation='vertical', pad=0.02, shrink=0.65, aspect=20)
cbar.set_label('Hist. events before quiescence', fontsize=8)
cbar.ax.tick_params(labelsize=7)

# Tectonic zone labels
tect_labels = [
    ('Sunda Megathrust', 2.0, 100.0),
    ('Banda Arc', -7.5, 126.5),
    ('Molucca Sea', 1.5, 126.5),
    ('New Guinea Trench', -4.0, 138.5),
]
for lbl, lat, lon in tect_labels:
    ax.text(lon, lat, lbl, transform=ccrs.PlateCarree(), fontsize=7,
            color='#1A5276', fontstyle='italic', ha='center',
            bbox=dict(boxstyle='round,pad=0.15', facecolor='white', alpha=0.6, edgecolor='none'))

gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
gl.top_labels = False; gl.right_labels = False
gl.xlabel_style = {'size': 8}; gl.ylabel_style = {'size': 8}

n_sp = len(spatial_gaps)
ax.set_title(f'Seismic Gap Map — {n_sp} Spatial Gaps Identified (Red Boxes)\nTop 10 Critical Gaps Highlighted with Thick Border',
             fontsize=10, fontweight='bold', pad=6)

plt.tight_layout()
plt.savefig(FIGDIR / 'figure4_seismic_gap_map.png', dpi=300)
plt.show()
print('[OK] figure4_seismic_gap_map.png')

## Cell 6: Figure 5 — LSTM Predictions (5 panels, zoom 2015-2026)

In [ ]:
# Load predictions, plot actual vs predicted with shading
# Boundaries: train end 2019-12, val end 2022-12, test start 2023-01
# Episodic note for Maluku & Papua
# Annotate Pearson and R2

EPISODIC = {'Zona Maluku', 'Zona Papua'}
fig, axes = plt.subplots(5, 1, figsize=(7.48, 10.0), sharex=False)
fig.suptitle('LSTM v3 Predictions vs Actual Monthly Earthquake Count\n(Test Period: Jan 2023 - May 2026)',
             fontsize=11, fontweight='bold')

for i, z in enumerate(ZONES):
    ax  = axes[i]
    col = ZCOLORS[z]
    p   = preds[z]

    # Zoom to 2015-2026
    mask  = p['date'] >= pd.Timestamp('2015-01-01')
    p_vis = p[mask]

    ax.fill_between(p_vis['date'], p_vis['actual'], p_vis['predicted'],
                    alpha=0.15, color=col)
    ax.plot(p_vis['date'], p_vis['actual'],    color='black', lw=1.4, alpha=0.9, label='Actual')
    ax.plot(p_vis['date'], p_vis['predicted'], color=col,     lw=1.4, ls='--',   label='Predicted v3')

    # Boundaries
    ax.axvline(pd.Timestamp('2023-01-01'), color='red',   ls='--', lw=1, alpha=0.6, label='Test start')
    ax.axvline(pd.Timestamp('2020-01-01'), color='green', ls=':',  lw=1, alpha=0.5, label='Val start')

    # Shading: val and test
    ax.axvspan(pd.Timestamp('2020-01-01'), pd.Timestamp('2022-12-31'), alpha=0.05, color='green')
    ax.axvspan(pd.Timestamp('2023-01-01'), pd.Timestamp('2026-06-01'), alpha=0.07, color='red')

    # Metrics from metrics df
    m_row = metrics[metrics['zone_name'] == z].iloc[0]
    r2_v  = m_row['r2']
    ps_v  = m_row['pearson']
    ann_text = f"Pearson={ps_v:.3f}  R2={r2_v:.3f}"
    ax.text(0.02, 0.92, ann_text, transform=ax.transAxes, fontsize=7.5,
            color=col, fontweight='bold', va='top',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8, edgecolor=col, lw=0.6))

    if z in EPISODIC:
        ax.text(0.98, 0.92, 'Episodic zone — stochastic behavior',
                transform=ax.transAxes, fontsize=7, ha='right', va='top',
                color='#7F8C8D', fontstyle='italic',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='#FDFEFE', alpha=0.8, edgecolor='#BDC3C7', lw=0.5))

    ax.set_title(ZSHORT[z], fontsize=10, color=col, fontweight='bold', loc='left')
    ax.set_ylabel('Events/month', fontsize=8)
    ax.set_xlim(pd.Timestamp('2015-01-01'), pd.Timestamp('2026-06-01'))
    ax.grid(alpha=0.2)
    if i == 0:
        ax.legend(fontsize=7, ncol=4, loc='upper right')

axes[-1].set_xlabel('Year')
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(FIGDIR / 'figure5_lstm_prediction.png', dpi=300)
plt.show()
print('[OK] figure5_lstm_prediction.png')

## Cell 7: Figure 6 — Composite Risk Map (cartopy + 1x1 degree grid + inset)

In [ ]:
# Build composite hazard per zone:
# composite = 0.4*hazard_score + 0.4*gap_norm + 0.2*pearson_norm
# Map each 1x1 degree cell to dominant zone -> color by composite score

gap_max = summary['gap_area_deg2'].max()
hazard_scores = dict(zip(summary['zone_name'], summary['hazard_score']))
gap_norms     = dict(zip(summary['zone_name'], summary['gap_area_deg2'] / gap_max))
pearson_norms = dict(zip(metrics['zone_name'], (metrics['pearson'] + 1) / 2))

composite = {}
for z in ZONES:
    h = hazard_scores.get(z, 0.0)
    g = gap_norms.get(z, 0.0)
    p_n = pearson_norms.get(z, 0.0)
    composite[z] = 0.4 * h + 0.4 * g + 0.2 * p_n
    print(f'  {ZSHORT[z]:<18}: hazard={h:.3f}  gap_norm={g:.3f}  pearson_norm={p_n:.3f}  composite={composite[z]:.3f}')

# Build 1x1 degree grid -> dominant zone per cell
lon_edges = np.arange(94, 143, 1)
lat_edges = np.arange(-13, 11, 1)

grid_data = {}
for lon0 in lon_edges[:-1]:
    for lat0 in lat_edges[:-1]:
        mask = ((df['longitude'] >= lon0) & (df['longitude'] < lon0+1) &
                (df['latitude']  >= lat0) & (df['latitude']  < lat0+1))
        sub = df[mask]
        if len(sub) < 2:
            continue
        dom_zone = sub['zone_name'].value_counts().idxmax()
        grid_data[(lon0, lat0)] = composite[dom_zone]

# Plot
fig = plt.figure(figsize=(7.48, 5.2))
ax  = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
ax.set_extent([94, 142, -12, 10], crs=ccrs.PlateCarree())

ax.add_feature(cfeature.OCEAN, facecolor='#D6EAF8', zorder=0)
ax.add_feature(cfeature.LAND,  facecolor='#F5F5F0', zorder=1)
ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='#333333', zorder=5)
ax.add_feature(cfeature.BORDERS, linewidth=0.3, edgecolor='#777777', linestyle='--', zorder=5)

cmap_risk = cm.get_cmap('RdYlGn_r')
norm_risk  = Normalize(vmin=0.2, vmax=0.9)

for (lon0, lat0), val in grid_data.items():
    color = cmap_risk(norm_risk(val))
    rect  = mpatches.Rectangle((lon0, lat0), 1, 1,
                                linewidth=0, facecolor=color, alpha=0.65,
                                transform=ccrs.PlateCarree(), zorder=3)
    ax.add_patch(rect)

# Zone boundary contours (outline per zone using cluster convex hull approximation)
_label_pos = {
    'Zona Jawa-Bali-NTB': (112.0, -9.5),
    'Zona Sulawesi-NTT':  (122.0, -6.5),
    'Zona Maluku':        (127.5, -2.0),
    'Zona Papua':         (134.0,  1.5),
}
for z in ZONES:
    sub = df[df['zone_name'] == z]
    col = ZCOLORS[z]
    if z in _label_pos:
        cx, cy = _label_pos[z]
    else:
        cx = float(sub['longitude'].median())
        cy = float(sub['latitude'].median())
    comp_val = composite[z]
    zone_lbl = ZSHORT[z] + f'\n({comp_val:.2f})'
    ax.text(cx, cy, zone_lbl,
            transform=ccrs.PlateCarree(), fontsize=7.5,
            fontweight='bold', color='white', ha='center', va='center', zorder=6,
            bbox=dict(boxstyle='round,pad=0.25', facecolor=ZCOLORS[z], alpha=0.85, edgecolor='white', lw=0.8))

# Colorbar
sm = cm.ScalarMappable(cmap=cmap_risk, norm=norm_risk)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, orientation='vertical', pad=0.02, shrink=0.7, aspect=20)
cbar.set_label('Composite Seismic Hazard Index', fontsize=9)
cbar.ax.tick_params(labelsize=8)

# Major cities
cities = {'Jakarta': (-6.21, 106.85), 'Surabaya': (-7.25, 112.75),
          'Medan': (3.59, 98.67), 'Makassar': (-5.14, 119.42), 'Jayapura': (-2.53, 140.72)}
for city, (lat, lon) in cities.items():
    ax.plot(lon, lat, 'w^', ms=5, markeredgecolor='black', markeredgewidth=0.5,
            transform=ccrs.PlateCarree(), zorder=7)
    ax.text(lon+0.3, lat+0.4, city, transform=ccrs.PlateCarree(),
            fontsize=6.5, color='black', zorder=7)

# Inset: SE Asia overview
ax_ins = fig.add_axes([0.02, 0.02, 0.22, 0.22], projection=ccrs.PlateCarree())
ax_ins.set_extent([80, 165, -20, 30], crs=ccrs.PlateCarree())
ax_ins.add_feature(cfeature.OCEAN, facecolor='#D6EAF8')
ax_ins.add_feature(cfeature.LAND,  facecolor='#E8E8E0')
ax_ins.add_feature(cfeature.COASTLINE, linewidth=0.3, edgecolor='#555555')
rect_ins = mpatches.Rectangle((94, -12), 48, 22, linewidth=1.5,
                               edgecolor='red', facecolor='none',
                               transform=ccrs.PlateCarree())
ax_ins.add_patch(rect_ins)
ax_ins.set_title('SE Asia', fontsize=6, pad=2)
ax_ins.tick_params(labelsize=5, length=2)

gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
gl.top_labels = False; gl.right_labels = False
gl.xlabel_style = {'size': 8}; gl.ylabel_style = {'size': 8}

ax.set_title('Composite Seismic Hazard Risk Map — Indonesia\n'
             'Index = 0.4xHazard + 0.4xGap + 0.2xLSTM Pearson (normalized)',
             fontsize=10, fontweight='bold', pad=6)

plt.subplots_adjust(left=0.05, right=0.88, top=0.92, bottom=0.25)

# Tabel di luar axes peta — axes terpisah di area bawah kanan figure
ax_table = fig.add_axes([0.35, 0.01, 0.40, 0.20])
ax_table.axis('off')

table_data = [
    ["Zona",          "Hazard", "Gap", "r"],
    ["Sumatera",      "0.582",  "33",  "0.74"],
    ["Jawa-Bali-NTB", "0.292",  "8",   "0.51"],
    ["Sulawesi-NTT",  "0.223",  "3",   "0.31"],
    ["Maluku",        "0.988",  "4",   "-0.05"],
    ["Papua",         "0.353",  "0",   "-0.05"],
]

tbl = ax_table.table(
    cellText=table_data[1:],
    colLabels=table_data[0],
    loc='center',
    cellLoc='center',
    bbox=[0, 0, 1, 1]
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.3, 1.4)
tbl.auto_set_column_width(col=list(range(4)))

# Header style
for j in range(4):
    tbl[(0, j)].set_facecolor("#1F4E79")
    tbl[(0, j)].set_text_props(color="white", fontweight="bold")
    tbl[(0, j)].set_edgecolor("white")

# Data rows alternating color
colors = ["#FFFFFF", "#EBF3FB"]
for i in range(1, 6):
    for j in range(4):
        tbl[(i, j)].set_facecolor(colors[i % 2])
        tbl[(i, j)].set_edgecolor("#CCCCCC")

ax_table.set_title("Hazard Summary", fontsize=10, fontweight='bold', pad=4, loc='left')

plt.savefig(FIGDIR / 'figure6_risk_map.png', dpi=300)
plt.show()
print('[OK] figure6_risk_map.png')
print('Composite scores:')
for z in ZONES:
    print(f'  {ZSHORT[z]}: {composite[z]:.4f}')
print('Figure 6 revised — tabel fixed ✓')

## Cell 8: Tables (4 CSVs)

In [ ]:
# Table 1: Dataset Summary
tbl1_rows = []
for z in ZONES:
    sub = df[df['zone_name'] == z]
    tbl1_rows.append({
        'Zona': ZSHORT[z],
        'N_Events': len(sub),
        'Percentage': round(len(sub)/len(df)*100, 2),
        'Lon_Range': f"{sub['longitude'].min():.1f} - {sub['longitude'].max():.1f}",
        'Depth_Mean': round(sub['depth'].mean(), 1),
        'Depth_Std':  round(sub['depth'].std(), 1),
        'Mag_Mean':   round(sub['mag'].mean(), 3),
        'Mag_Max':    round(sub['mag'].max(), 2),
    })
tbl1 = pd.DataFrame(tbl1_rows)
tbl1.to_csv(TABDIR / 'table1_dataset_summary.csv', index=False)
print('Table 1:'); print(tbl1.to_string(index=False))

# Table 2: Clustering Results (with silhouette score)
from sklearn.metrics import silhouette_score as sil_score
feat_cols = ['latitude', 'longitude', 'depth', 'mag']
df_clean  = df.dropna(subset=feat_cols + ['cluster'])
sil = sil_score(df_clean[feat_cols], df_clean['cluster'], sample_size=10000, random_state=42)
print(f'\nGlobal Silhouette Score: {sil:.4f}')

tect_map = {
    'Zona Sumatera':      'Sunda Subduction Zone',
    'Zona Jawa-Bali-NTB': 'Java-Bali Subduction',
    'Zona Sulawesi-NTT':  'Sulawesi-Banda Arc',
    'Zona Maluku':        'Molucca Sea Collision',
    'Zona Papua':         'New Guinea Thrust Belt',
}
tbl2_rows = []
for z in ZONES:
    row = summary[summary['zone_name'] == z].iloc[0]
    tbl2_rows.append({
        'Zona': ZSHORT[z],
        'N_Events': int(row['n_event']),
        'Silhouette_Score': round(sil, 4),
        'Depth_Mean': round(df[df['zone_name']==z]['depth'].mean(), 1),
        'Mag_Mean':   round(float(row['mean_mag']), 3),
        'Tectonic_Setting': tect_map[z],
    })
tbl2 = pd.DataFrame(tbl2_rows)
tbl2.to_csv(TABDIR / 'table2_clustering_results.csv', index=False)
print('\nTable 2:'); print(tbl2.to_string(index=False))

# Table 3: Seismic Hazard Assessment — compute Mc per zone
tbl3_rows = []
for z in ZONES:
    row   = summary[summary['zone_name'] == z].iloc[0]
    mags  = df[df['zone_name'] == z]['mag'].dropna().values
    gr    = compute_gr_params(mags)
    n_sp  = len(gaps_df[(gaps_df['zone_name']==z) & (gaps_df['gap_type']=='spatial')])
    tbl3_rows.append({
        'Zona': ZSHORT[z],
        'b_value': round(float(row['b_value']), 4),
        'Mc': round(gr['mc'], 1),
        'Recurrence_Interval_days': round(float(row['recurrence_interval_days']), 1),
        'Max_Mag': round(float(row['max_mag']), 2),
        'N_Spatial_Gaps': n_sp,
        'Hazard_Score': round(float(row['hazard_score']), 4),
        'Hazard_Category': row['hazard_category'],
    })
tbl3 = pd.DataFrame(tbl3_rows)
tbl3.to_csv(TABDIR / 'table3_hazard_assessment.csv', index=False)
print('\nTable 3:'); print(tbl3.to_string(index=False))

# Table 4: LSTM Performance
# Compute hit_rate from predictions
def compute_hit_rate(p_df, threshold=0.20):
    denom = np.maximum(np.abs(p_df['actual'].values), 1e-6)
    hit = np.abs(p_df['actual'].values - p_df['predicted'].values) <= threshold * denom
    return round(float(hit.mean() * 100), 2)

tect_impl = {
    'Zona Sumatera':      'High seismicity rate, aftershock-dominated',
    'Zona Jawa-Bali-NTB': 'Moderate predictability, regular recurrence',
    'Zona Sulawesi-NTT':  'Deep slab dynamics, low event rate',
    'Zona Maluku':        'Episodic, multi-fault interaction',
    'Zona Papua':         'Complex collision zone, stochastic',
}
tbl4_rows = []
for z in ZONES:
    m_row = metrics[metrics['zone_name'] == z].iloc[0]
    hr    = compute_hit_rate(preds[z])
    r2_v  = m_row['r2']
    if r2_v >= 0.5:   assess = 'Excellent'
    elif r2_v >= 0.3: assess = 'Good'
    elif r2_v >= 0.0: assess = 'Acceptable'
    else:             assess = 'Poor'
    tbl4_rows.append({
        'Zona':                ZSHORT[z],
        'RMSE':                round(float(m_row['rmse']), 4),
        'MAE':                 round(float(m_row['mae']),  4),
        'MAPE':                round(float(m_row['mape']), 2),
        'R2':                  round(float(m_row['r2']),   4),
        'Pearson':             round(float(m_row['pearson']), 4),
        'HitRate_pct':         hr,
        'Model_Assessment':    assess,
        'Tectonic_Implication': tect_impl[z],
    })
tbl4 = pd.DataFrame(tbl4_rows)
tbl4.to_csv(TABDIR / 'table4_lstm_performance.csv', index=False)
print('\nTable 4:'); print(tbl4.to_string(index=False))

## Cell 9: Export Summary

In [ ]:
sep = '=' * 70
print(sep)
print('EXPORT SUMMARY — 04_summary_jurnal.ipynb')
print(sep)

all_outputs = {
    'figures': [
        FIGDIR / 'figure1_seismicity_map.png',
        FIGDIR / 'figure2_temporal_analysis.png',
        FIGDIR / 'figure3_gutenberg_richter.png',
        FIGDIR / 'figure4_seismic_gap_map.png',
        FIGDIR / 'figure5_lstm_prediction.png',
        FIGDIR / 'figure6_risk_map.png',
    ],
    'tables': [
        TABDIR / 'table1_dataset_summary.csv',
        TABDIR / 'table2_clustering_results.csv',
        TABDIR / 'table3_hazard_assessment.csv',
        TABDIR / 'table4_lstm_performance.csv',
    ],
}
grand_total_kb = 0
all_ok = True

for category, paths in all_outputs.items():
    n_cat   = len(paths)
    n_found = sum(1 for p in paths if p.exists())
    print(f'\n  {category.upper()} ({n_found}/{n_cat} files):')
    for fp in paths:
        if fp.exists():
            kb = fp.stat().st_size / 1024
            grand_total_kb += kb
            print(f'    [OK]  {fp.name:<50} {kb:>7.1f} KB')
        else:
            print(f'    [MISSING] {fp.name}')
            all_ok = False

print(f'\n{sep}')
print(f'Total figures : {len(all_outputs["figures"])}')
print(f'Total tables  : {len(all_outputs["tables"])}')
total_mb = grand_total_kb / 1024
print(f'Total size    : {total_mb:.2f} MB')
print(f'Status        : {"ALL OK" if all_ok else "SOME FILES MISSING"}')
print(sep)
print('\nFigure specs:')
print('  DPI     : 300')
print('  Width   : 7.48 in (190 mm, double-column)')
print('  Font    : Arial / DejaVu Sans, 11pt base')
print('  Format  : PNG, white background')
print(sep)